## Q2. Which types of company and which warning signs carry the most risk?
Purpose: compare borrower segments, then warning signs within trading companies, on an age-adjusted basis; plus current distress as a red flag.
Method: for each group, chance of deregistering within 5 years = 1 − S(5), built from the group's own rates in age bands 0–1 … 4–5 (so groups with different age mixes are compared fairly).
Relative risk index (RRI) = group's 1 − S(5) ÷ benchmark's 1 − S(5). Above 1.25 = higher risk, below 0.80 = lower risk (illustrative thresholds).
Reliability: a group needs all 5 age bands with ≥ 100 company-years and ≥ 10 deregistrations each, otherwise "Not enough data".
Tables: fct_exposure, fct_company_snapshot, dim_company_classification, dim_company, dim_age_band.

In [0]:
%sql
-- Step 1: exposure and deregistrations per segment and age band (first 5 years)
WITH band_totals AS (
    SELECT
        k.borrower_segment       AS group_name,
        k.segment_sort_order     AS group_order,
        f.age_band_key,
        SUM(f.exposure_years)    AS exposure_years,
        SUM(f.deregistrations)   AS deregistrations
    FROM workspace.sme_risk_gold.fct_exposure f
    JOIN workspace.sme_risk_gold.dim_company_classification k
        ON f.classification_key = k.classification_key
    WHERE f.age_band_key BETWEEN 1 AND 5
    GROUP BY k.borrower_segment, k.segment_sort_order, f.age_band_key

    UNION ALL

    SELECT 'All companies', 0, f.age_band_key, SUM(f.exposure_years), SUM(f.deregistrations)
    FROM workspace.sme_risk_gold.fct_exposure f
    WHERE f.age_band_key BETWEEN 1 AND 5
    GROUP BY f.age_band_key
),

-- Step 2: 5-year survival per group, plus reliability inputs
group_survival AS (
    SELECT
        group_name,
        group_order,
        COUNT(*)                                        AS bands_with_data,
        MIN(exposure_years)                             AS min_band_exposure,
        MIN(deregistrations)                            AS min_band_deregistrations,
        SUM(exposure_years)                             AS exposure_years,
        SUM(deregistrations)                            AS deregistrations,
        EXP(-SUM(deregistrations / exposure_years))     AS survival_5y
    FROM band_totals
    GROUP BY group_name, group_order
),

-- Step 3: risk index vs the benchmark (all companies)
scored AS (
    SELECT
        *,
        (1 - survival_5y)
          / MAX(CASE WHEN group_name = 'All companies' THEN 1 - survival_5y END) OVER () AS rri,
        bands_with_data = 5 AND min_band_exposure >= 100 AND min_band_deregistrations >= 10 AS is_reliable
    FROM group_survival
)

SELECT
    group_name                                AS borrower_segment,
    ROUND(exposure_years, 0)                  AS company_years_age_0_5,
    deregistrations                           AS deregistrations_age_0_5,
    ROUND(100 * survival_5y, 1)               AS survival_5y_pct,
    ROUND(100 * (1 - survival_5y), 1)         AS dereg_within_5y_pct,
    CASE WHEN is_reliable THEN ROUND(rri, 2) END AS risk_index,
    CASE
        WHEN NOT is_reliable THEN 'Not enough data'
        WHEN rri > 1.25      THEN 'Higher risk'
        WHEN rri < 0.80      THEN 'Lower risk'
        ELSE 'Around average'
    END                                       AS risk_flag
FROM scored
ORDER BY group_order;

Databricks visualization. Run in Databricks to view.

In [0]:
%sql
-- Step 1: trading-company exposure rows with their company attributes
WITH trading AS (
    SELECT f.age_band_key, f.exposure_years, f.deregistrations, c.has_abn, c.has_name_change, c.is_acn_named
    FROM workspace.sme_risk_gold.fct_exposure f
    JOIN workspace.sme_risk_gold.dim_company_classification k ON f.classification_key = k.classification_key
    JOIN workspace.sme_risk_gold.dim_company c                ON f.company_key        = c.company_key
    WHERE k.borrower_segment = 'Trading company'
      AND f.age_band_key BETWEEN 1 AND 5
),

-- Step 2: one set of rows per warning sign (each company appears once per dimension)
labelled AS (
    SELECT 0 AS dim_order, 'Benchmark'   AS dimension, 'All trading companies' AS group_name, age_band_key, exposure_years, deregistrations FROM trading
    UNION ALL
    SELECT 1, 'ABN held',     CASE WHEN has_abn         THEN 'Has ABN'                ELSE 'No ABN'        END, age_band_key, exposure_years, deregistrations FROM trading
    UNION ALL
    SELECT 2, 'Name history', CASE WHEN has_name_change THEN 'Renamed'                ELSE 'Never renamed' END, age_band_key, exposure_years, deregistrations FROM trading
    UNION ALL
    SELECT 3, 'Name type',    CASE WHEN is_acn_named    THEN 'Numbered (ACN in name)' ELSE 'Named'         END, age_band_key, exposure_years, deregistrations FROM trading
),

-- Step 3: totals per group and age band
band_totals AS (
    SELECT dim_order, dimension, group_name, age_band_key,
           SUM(exposure_years) AS exposure_years, SUM(deregistrations) AS deregistrations
    FROM labelled
    GROUP BY dim_order, dimension, group_name, age_band_key
),

-- Step 4: 5-year survival per group
group_survival AS (
    SELECT dim_order, dimension, group_name,
           COUNT(*)                                    AS bands_with_data,
           MIN(exposure_years)                         AS min_band_exposure,
           MIN(deregistrations)                        AS min_band_deregistrations,
           SUM(exposure_years)                         AS exposure_years,
           SUM(deregistrations)                        AS deregistrations,
           EXP(-SUM(deregistrations / exposure_years)) AS survival_5y
    FROM band_totals
    GROUP BY dim_order, dimension, group_name
),

-- Step 5: risk index vs all trading companies
scored AS (
    SELECT *,
           (1 - survival_5y)
             / MAX(CASE WHEN group_name = 'All trading companies' THEN 1 - survival_5y END) OVER () AS rri,
           bands_with_data = 5 AND min_band_exposure >= 100 AND min_band_deregistrations >= 10   AS is_reliable
    FROM group_survival
)

SELECT
    dimension,
    group_name,
    ROUND(exposure_years, 0)                      AS company_years_age_0_5,
    deregistrations                               AS deregistrations_age_0_5,
    ROUND(100 * (1 - survival_5y), 1)             AS dereg_within_5y_pct,
    CASE WHEN is_reliable THEN ROUND(rri, 2) END  AS risk_index,
    CASE
        WHEN NOT is_reliable THEN 'Not enough data'
        WHEN rri > 1.25      THEN 'Higher risk'
        WHEN rri < 0.80      THEN 'Lower risk'
        ELSE 'Around average'
    END                                           AS risk_flag
FROM scored
ORDER BY dim_order, group_name;

Databricks visualization. Run in Databricks to view.

In [0]:
%sql
SELECT
    k.borrower_segment,
    a.age_group,
    SUM(f.is_active)                                           AS active_companies,
    SUM(f.is_distressed)                                       AS distressed,
    ROUND(100.0 * SUM(f.is_distressed) / SUM(f.is_active), 2)  AS distress_share_pct
FROM workspace.sme_risk_gold.fct_company_snapshot f
JOIN workspace.sme_risk_gold.dim_company_classification k ON f.classification_key = k.classification_key
JOIN workspace.sme_risk_gold.dim_age_band a               ON f.age_band_key       = a.age_band_key
WHERE f.is_active = 1
  AND k.borrower_segment IN ('Trading company', 'Super trustee', 'Not-for-profit')
GROUP BY k.borrower_segment, k.segment_sort_order, a.age_group
ORDER BY k.segment_sort_order, a.age_group;